# Calculo de Tendencias por niveles
***
Created: 17/04/2026                                    updated:01/05/2026


En este notebook se calcula la tendencia por pixeles y para profundides de entre 4000 y el fondo oceánico de forma que cada 20 metros de profundidad se tiene una tendencia. Además, se calculan otros datos necesarios para el calculo del calor, como densidad, area y capacidad calorifica. Finalmente, se calcula también el calor. Por ello, la idea ahora es generar un archivo netcdf que contenga un dataset con las siguientes carácteristicas:
- Dimensiones: (latitud, longitud, presion)
- variables:
    - $tendency \in \mathbb{R}^{lat \times lon\times press}$

Por defecto las fechas en las que se recortan es entre 1990 y 2010, además de profundidades inferiores a 4000 metros, pero esto se puede ajustar al principio.

La lógica del script es la siguiente:
1. Se abren los ficheros grid, join y batimetria
2. Se recortan para las fechas y las profundidades dichas
3. Se crea una función que dada ambos ds recortados devuelva las matrices de latitud, longitud, presión, tendencia media
4. Se guardan dichas matrices en un nuevo fichero netcdf


In [1]:
# Packages for data manipulation
import numpy as np
import xarray as xr
import pandas as pd
import datetime as dt

# Package for tendency calculation
from scipy import stats

# Packages for visualization
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Package for progress bar
from tqdm import tqdm

# Package for file handling
import os

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Características previas

In [29]:
grid_resolution = 1
min_press = 4000
max_press = None
start_year = 1990
end_year = 2010
press_step = 20
code_press =str(min_press)[0] + 'k'
robust = False
min_years = 2.5
group_by_years = False

### Apertura y recorte de los archivos

In [30]:
grid_res_str = str(grid_resolution).split('.')[-1]
grid = xr.open_dataset(f'./Data/grid/occupation_grid_{grid_res_str}.nc')
ds = xr.open_dataset('./Data/join/total_filt.nc')

In [ ]:
grid

In [ ]:
ds

Recortamos ambos datasets por fechas y profundidades

In [31]:
# Cutting the dataset to have data from the wanted years
grid = grid.where((grid.times.dt.year >= start_year) & (grid.times.dt.year <= end_year), drop = True)

In [32]:
# Cutting the dataset to have data from exact pressure
if max_press is not None:
    ds = ds.where((ds.pressure <= max_press) & (ds.pressure >= min_press), drop=True)

else:
    ds = ds.where(ds.pressure >= min_press, drop=True)

### Función que devuelve tendencias para el nuevo dataset

In [33]:
def tendencias_nivel(ds, grid, press_step = 20, robust = False, min_years = 10, group_by_years = False):
    # Extract the wanted values
    latitude = grid.latitude.values
    longitude = grid.longitude.values
    pressure = ds.pressure.values
    n = grid.n.values
    profiles = grid.profiles.values
    time_array = ds.time.values

    # Create the array of pressures with the wanted step
    levels = np.arange(np.min(pressure), np.max(pressure), press_step)

    # Array which contain tendency for levels
    tendency_levels = np.full((len(latitude), len(longitude), len(levels)), np.nan)

    # Precalculating the valid indeces
    valid_idx = []
    for i in range(len(latitude)):
        for j in range(len(longitude)):
            # Check if there are profiles
            if not np.isnan(n[i, j]): 
                # Obtaining index of profiles with Nans
                profs = profiles[i, j, :] 
                profs = profs[~np.isnan(profs)].astype(int)

                # Precalculate the dates for this pixel
                dates = time_array[profs]
                valid_idx.append((i, j, profs, dates))

    for k in range(1, len(levels)):
        # Cut the matrix by levels
        ds_level = ds.isel(P = ((ds.pressure < levels[k]) & (ds.pressure >= levels[k-1])), drop=True)

        # Calculate the mean for all the profiles
        temp_mean_profs = ds_level.ctd_temperature_filt.mean(dim = 'P').values

        # Explore all the valid data
        for i, j, profs, prof_dates in valid_idx:
            # Extract the mean
            temp_mean = temp_mean_profs[profs]

            # Delete Nans
            valid = ~np.isnan(temp_mean)
            temp_mean = temp_mean[valid]
            valid_dates = prof_dates[valid]

            if group_by_years:
                years = valid_dates.astype('datetime64[Y]')
                unique_years = np.unique(years)


                # Check if there are more than 2 values
                if len(unique_years) <= 2:
                    continue 
                
                # Calculate the anual mean
                anual_temp = np.array([np.mean(temp_mean[years == y]) for y in unique_years])
                annual_dates = unique_years.astype('datetime64[ns]')


                # Check if the difference between dates is longer than 2.5 years
                sorted_dates = np.sort(valid_dates)
                if (sorted_dates[-1] - sorted_dates[0]) / np.timedelta64(365, 'D') >= min_years:
                    # Fit to a line curve
                    dates_pol = annual_dates.astype(np.int64)
                    if robust:
                        coefficients = stats.theilslopes(anual_temp, dates_pol) # Ajust by square minimun
                    else:
                        coefficients = np.polyfit(dates_pol, anual_temp, 1)

                    tendency = coefficients[0] / 1.e-9 * 24 * 3600 * 365 * 100 # Convert from ºC/ns to ºC/century

                    # Save the value
                    tendency_levels[i, j, k] = tendency

            else:
                # Check if there are more than 2 values
                if len(valid_dates) <= 2:
                    continue 
                
                # Check if the difference between dates is longer than 2.5 years
                sorted_dates = np.sort(valid_dates)
                if (sorted_dates[-1] - sorted_dates[0]) / np.timedelta64(365, 'D') >= 2.5:
                    # Fit to a line curve
                    dates_pol = valid_dates.astype('datetime64[ns]').astype(np.int64)
                    if robust:
                        coefficients = stats.theilslopes(temp_mean, dates_pol) # Ajust by square minimun
                    else:
                        coefficients = np.polyfit(dates_pol, temp_mean, 1)

                    tendency = coefficients[0] / 1.e-9 * 24 * 3600 * 365 * 100 # Convert from ºC/ns to ºC/century          
                    # Save the value
                    tendency_levels[i, j, k] = tendency


            

                
    return (latitude, longitude, levels, tendency_levels)

Aplicamos y guardamos en un netcdf

In [34]:
# Extract matrix of data
lat, lon, press_levels, tendency_levels = tendencias_nivel(ds = ds, grid = grid, press_step = press_step, robust = robust, min_years = min_years, group_by_years = group_by_years) 

In [35]:
# Creating the dataset
ds_tendency = xr.Dataset(
    # Variables
    data_vars = {'tendency' : (('latitude', 'longitude', 'pressure'), tendency_levels, {'units': 'ºC/century', 'description': 'Temperature tendency by levels between the asociated pressure value and the one before'})},

    # Coordinates
    coords = {
        'latitude' : (('latitude',), lat ),
        'longitude' : (('longitude',), lon),
        'pressure': (('pressure',), press_levels, {'units' : 'dbar'}),
        'mask' : (('latitude', 'longitude'), grid.mask.values)},

    # Metadates
    attrs = {'description' : f'Dataset of temperature tendencies by pressure levels between from pressures under {min_press} dbar and for years {start_year}-{end_year} with a spatial resolution of {grid_resolution} and a pressure resolution {press_step} dbar'}
)

# Save it in a NetCDF file
if robust:
    ds_tendency.to_netcdf(f'./Data/tendency_levels/tendency_levels_{start_year}_{end_year}_{str(grid_resolution).split('.')[-1]}_{code_press}_robust.nc')
else:
    ds_tendency.to_netcdf(f'./Data/tendency_levels/tendency_levels_{start_year}_{end_year}_{str(grid_resolution).split('.')[-1]}_{code_press}.nc')

In [36]:
ds_tendency

<xarray.Dataset> Size: 51MB
Dimensions:    (latitude: 141, longitude: 358, pressure: 125)
Coordinates:
  * latitude   (latitude) float64 1kB -77.0 -76.0 -75.0 -74.0 ... 61.0 62.0 63.0
  * longitude  (longitude) float64 3kB -180.0 -179.0 -178.0 ... 179.0 180.0
    mask       (latitude, longitude) float64 404kB 10.0 10.0 10.0 ... nan nan
  * pressure   (pressure) int64 1kB 4000 4020 4040 4060 ... 6420 6440 6460 6480
Data variables:
    tendency   (latitude, longitude, pressure) float64 50MB nan nan ... nan nan
Attributes:
    description:  Dataset of temperature tendencies by pressure levels betwee...

In [ ]:
# Close all ds
ds.close()
grid.close()
ds_tendency.close()

: 